In [1]:
import torchvision.transforms as transforms
import torch
import torch.nn.functional as F
from torchvision import transforms
from torch.utils.data import DataLoader, Subset, ConcatDataset
from torch.optim import SGD, lr_scheduler
from utils.utils import *
import numpy as np
import matplotlib.pyplot as plt
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print('GPU') if str(device) == "cuda:0" else print('GPU not Detected - CPU Selected')
print(f"GPUs Count: {torch.cuda.device_count()}") # Command to show how many GPUs are available

GPU
GPUs Count: 1


In [2]:
my_train_batch_size = 64

In [ ]:
from dataloaders.cifar10 import cifar10_dataloaders
from torchvision.datasets import MNIST, FashionMNIST, CIFAR10, CIFAR100
from dataloaders.tinyimagenet import tinyimagenet_dataloaders

def mnist_dataloaders(train_batch_size=my_train_batch_size):
    transform = transforms.Compose([transforms.Lambda(lambda img: img.convert('RGB')), transforms.ToTensor()])
    train_dataset = Subset(MNIST('./data', train=True, transform=transform, download=True), list(range(50000)))
    train_dataset = ConcatDataset([train_dataset, train_dataset])   ############################### Duplicate the Data_Loader
    test_dataset = MNIST(root='./data', train=False, download=True, transform=transform)
    train_loader = DataLoader(train_dataset, batch_size=train_batch_size, shuffle=True, drop_last=True)
    test_loader = DataLoader(test_dataset, batch_size=100, shuffle=False, drop_last=True)
    return train_loader, test_loader

def fashion_mnist_dataloaders(train_batch_size=my_train_batch_size):
    train_transform = transforms.Compose([transforms.RandomHorizontalFlip(),transforms.Lambda(lambda img: img.convert('RGB')), transforms.ToTensor()])
    test_transform = transforms.Compose([transforms.Lambda(lambda img: img.convert('RGB')), transforms.ToTensor()]) 
    train_dataset = Subset(FashionMNIST('./data', train=True, transform=train_transform, download=True), list(range(50000)))
    train_dataset = ConcatDataset([train_dataset, train_dataset])   ############################### Duplicate the Data_Loader
    test_dataset = FashionMNIST(root='./data', train=False, download=True, transform=test_transform)
    train_loader = DataLoader(train_dataset, batch_size=train_batch_size, shuffle=True, drop_last=True)
    test_loader = DataLoader(test_dataset, batch_size=100, shuffle=False, drop_last=True) 
    return train_loader, test_loader

def cifar10_dataloaders(train_batch_size=my_train_batch_size, data_dir = 'datasets/cifar10'):
    train_transform = transforms.Compose([transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip(), transforms.ToTensor()])
    test_transform = transforms.Compose([transforms.ToTensor()])
    train_dataset = Subset(CIFAR10(data_dir, train=True, transform=train_transform, download=True), list(range(50000)))
    train_dataset = ConcatDataset([train_dataset, train_dataset])   ############################### Duplicate the Data_Loader
    test_dataset = CIFAR10(data_dir, train=False, transform=test_transform, download=True)
    train_loader = DataLoader(train_dataset, batch_size=train_batch_size, shuffle=True, drop_last=True)
    test_loader = DataLoader(test_dataset, batch_size=100, shuffle=False, drop_last=True)
    return train_loader, test_loader

def cifar100_dataloaders(train_batch_size=128, data_dir = 'datasets/cifar100'):
    train_transform = transforms.Compose([transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip(), transforms.ToTensor()])
    test_transform = transforms.Compose([transforms.ToTensor()])
    train_dataset = Subset(CIFAR100(data_dir, train=True, transform=train_transform, download=True), list(range(50000)))
    train_dataset = ConcatDataset([train_dataset, train_dataset])   ############################### Duplicate the Data_Loader
    test_dataset = CIFAR100(data_dir, train=False, transform=test_transform, download=True)
    train_loader = DataLoader(train_dataset, batch_size=train_batch_size, shuffle=True, drop_last=True)
    test_loader = DataLoader(test_dataset, batch_size=100, shuffle=False, drop_last=True)
    return train_loader, test_loader

train_loader_1, test_loader_1 = mnist_dataloaders(train_batch_size=my_train_batch_size)
train_loader_2, test_loader_2 = fashion_mnist_dataloaders(train_batch_size=my_train_batch_size)
train_loader_3, test_loader_3 = cifar10_dataloaders(train_batch_size=my_train_batch_size)
train_loader_4, test_loader_4 = cifar100_dataloaders(train_batch_size=my_train_batch_size)
train_loader_5, test_loader_5 = tinyimagenet_dataloaders(train_batch_size=my_train_batch_size)

### All Data_Loaders other than TinyImageNet, have been **twofolded**

In [4]:
len(train_loader_1), len(train_loader_2), len(train_loader_3), len(train_loader_4), len(train_loader_5)

(1562, 1562, 1562, 1562, 1562)

In [5]:
len(test_loader_1), len(test_loader_2), len(test_loader_3), len(test_loader_4), len(test_loader_5)

(100, 100, 100, 100, 100)

### **UNIFIED Model**

In [6]:
my_classes = [10, 10, 10, 100, 200]

class MultiBN2d(nn.Module):
    def __init__(self, num_features):
        super(MultiBN2d, self).__init__()
        self.BN_1 = nn.BatchNorm2d(num_features)
        self.BN_2 = nn.BatchNorm2d(num_features)
        self.BN_3 = nn.BatchNorm2d(num_features)
        self.BN_4 = nn.BatchNorm2d(num_features)
        self.BN_5 = nn.BatchNorm2d(num_features)

    def forward(self, _input, dataset_idx):
        if dataset_idx == 1:
            _output = self.BN_1(_input)
        elif dataset_idx == 2:
            _output = self.BN_2(_input)
        elif dataset_idx == 3:
            _output = self.BN_3(_input)
        elif dataset_idx == 4:
            _output = self.BN_4(_input)
        elif dataset_idx == 5:
            _output = self.BN_5(_input)
        return _output

class BasicBlock(nn.Module):
    def __init__(self, in_planes, out_planes, stride, dropRate=0.0):
        super(BasicBlock, self).__init__()
        self.bn1 = MultiBN2d(in_planes)
        self.relu1 = nn.ReLU(inplace=True)
        self.conv1 = nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn2 = MultiBN2d(out_planes)
        self.relu2 = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_planes, out_planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.droprate = dropRate
        self.equalInOut = (in_planes == out_planes)
        self.convShortcut = (not self.equalInOut) and nn.Conv2d(in_planes, out_planes, kernel_size=1, stride=stride, padding=0, bias=False) or None
    
    def forward(self, x, dataset_idx):
        if not self.equalInOut:
            x = self.relu1(self.bn1(x, dataset_idx))
        else:
            out = self.relu1(self.bn1(x, dataset_idx))
        out = self.relu2(self.bn2(self.conv1(out if self.equalInOut else x), dataset_idx))
        if self.droprate > 0:
            out = F.dropout(out, p=self.droprate, training=self.training)
        out = self.conv2(out)
        return torch.add(x if self.equalInOut else self.convShortcut(x), out)

class NetworkBlock(nn.Module):
    def __init__(self, nb_layers, in_planes, out_planes, block, stride, dropRate=0.0):
        super(NetworkBlock, self).__init__()
        self.layers = nn.ModuleList()
        for i in range(int(nb_layers)):
            self.layers.append(
                block(
                    in_planes if i == 0 else out_planes,
                    out_planes,
                    stride if i == 0 else 1,
                    dropRate
                )
            )

    def forward(self, x, dataset_idx):
        for layer in self.layers:
            x = layer(x, dataset_idx)
        return x

class WideResNet(nn.Module):
    def __init__(self, depth, num_classes, widen_factor=1, dropRate=0.0):
        super(WideResNet, self).__init__()
        nChannels = [16, 16*widen_factor, 32*widen_factor, 64*widen_factor]
        assert((depth - 4) % 6 == 0)
        n = (depth - 4) / 6
        block = BasicBlock

        self.conv1 = nn.Conv2d(3, nChannels[0], kernel_size=3, stride=1, padding=1, bias=False)
        self.block1 = NetworkBlock(n, nChannels[0], nChannels[1], block, 1, dropRate)
        self.block2 = NetworkBlock(n, nChannels[1], nChannels[2], block, 2, dropRate)
        self.block3 = NetworkBlock(n, nChannels[2], nChannels[3], block, 2, dropRate)
        
        self.bn1 = MultiBN2d(nChannels[3])
        self.relu = nn.ReLU(inplace=True)
        #self.fc = nn.Linear(nChannels[3], num_classes)
        
        ####### Separate Output Heads
        self.output_heads = nn.ModuleDict({
            str(data_id + 1): nn.Linear(nChannels[3], c) for data_id, c in enumerate(num_classes)
        })
        
        self.nChannels = nChannels[3]

        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                m.weight.data.fill_(1)
                m.bias.data.zero_()
            elif isinstance(m, nn.Linear):
                m.bias.data.zero_()
                
    def forward(self, x, dataset_idx):
        out = self.conv1(x)
        out = self.block1(out, dataset_idx)
        out = self.block2(out, dataset_idx)
        out = self.block3(out, dataset_idx)
        out = self.relu(self.bn1(out, dataset_idx))
        out = F.avg_pool2d(out, kernel_size=out.shape[2:])
        out = out.view(-1, self.nChannels)
        #out = self.fc(out)
        out = self.output_heads[str(dataset_idx)](out)
        return out

In [ ]:
# Simulate argparse values
class Args:
    batch_size, dim = my_train_batch_size, my_train_batch_size
    epochs = 200
    lr = 0.1
    momentum = 0.9
    wd = 5e-4
args = Args()

############### MODEL SELECTION:
model = WideResNet(depth=16, num_classes=my_classes, widen_factor=8, dropRate=0.0).to(device)

##### optimizer:
optimizer = SGD(model.parameters(), lr=args.lr, momentum=args.momentum, weight_decay=args.wd)
scheduler = lr_scheduler.CosineAnnealingLR(optimizer, args.epochs)

In [8]:
model.output_heads

ModuleDict(
  (1): Linear(in_features=512, out_features=10, bias=True)
  (2): Linear(in_features=512, out_features=10, bias=True)
  (3): Linear(in_features=512, out_features=10, bias=True)
  (4): Linear(in_features=512, out_features=100, bias=True)
  (5): Linear(in_features=512, out_features=200, bias=True)
)

In [9]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params}")

torch.save(model.state_dict(), 'test_model.pth')

Total parameters: 11154330


### **Training Loop**

In [10]:
loss_fn = nn.CrossEntropyLoss()

In [ ]:
################################## TRAINING:
best_TA = 0

# Dictionaries to store curves data
data_plots = {"train_total_loss": [], "train_loss1": [], "train_loss2": [], "train_loss3": [], "train_loss4": [], "train_loss5": [], "train_acc1": [], "train_acc2": [], "train_acc3": [], "train_acc4": [], "train_acc5": [], "val_total_loss": [], "val_acc1": [], "val_acc2": [], "val_acc3": [], "val_acc4": [], "val_acc5": [], "val_loss1": [], "val_loss2": [], "val_loss3": [], "val_loss4": [], "val_loss5": []}

for epoch in range(1, args.epochs):
    model.train()
    requires_grad_(model, True)
    accs_1, accs_2, accs_3, accs_4, accs_5, losses_1, losses_2, losses_3, losses_4, losses_5, total_losses = AverageMeter(), AverageMeter(), AverageMeter(), AverageMeter(), AverageMeter(), AverageMeter(), AverageMeter(), AverageMeter(), AverageMeter(), AverageMeter(), AverageMeter()
    i = 0
    for (imgs_1, labels_1), (imgs_2, labels_2), (imgs_3, labels_3), (imgs_4, labels_4), (imgs_5, labels_5) in zip(train_loader_1, train_loader_2, train_loader_3, train_loader_4, train_loader_5):
        i+=1
        imgs_1, labels_1, imgs_2, labels_2, imgs_3, labels_3, imgs_4, labels_4, imgs_5, labels_5 = imgs_1.to(device), labels_1.to(device), imgs_2.to(device), labels_2.to(device), imgs_3.to(device), labels_3.to(device), imgs_4.to(device), labels_4.to(device), imgs_5.to(device), labels_5.to(device)
        optimizer.zero_grad()
        
        logits_1 = model(imgs_1, 1)
        loss_1 = loss_fn(logits_1, labels_1)
        
        logits_2 = model(imgs_2, 2)
        loss_2 = loss_fn(logits_2, labels_2)
        
        logits_3 = model(imgs_3, 3)
        loss_3 = loss_fn(logits_3, labels_3)
        
        logits_4 = model(imgs_4, 4)
        loss_4 = loss_fn(logits_4, labels_4)
        
        logits_5 = model(imgs_5, 5)
        loss_5 = loss_fn(logits_5, labels_5)
        
        ############################################### TOTAL Loss:    
        total_loss = loss_1 + loss_2 + loss_3 + loss_4 + loss_5
        total_loss.backward()
        optimizer.step()

        ##### metrics:
        accs_1.append((logits_1.argmax(1) == labels_1).float().mean().item() * 100)
        accs_2.append((logits_2.argmax(1) == labels_2).float().mean().item() * 100)
        accs_3.append((logits_3.argmax(1) == labels_3).float().mean().item() * 100)
        accs_4.append((logits_4.argmax(1) == labels_4).float().mean().item() * 100)
        accs_5.append((logits_5.argmax(1) == labels_5).float().mean().item() * 100)
        losses_1.append(loss_1.item())
        losses_2.append(loss_2.item())
        losses_3.append(loss_3.item())
        losses_4.append(loss_4.item())
        losses_5.append(loss_5.item())
        total_losses.append(total_loss.item())
        if i % 500 == 0:
            train_str = 'Epoch %d-%d | Loss: %.4f,  Acc_1: %.2f,  Acc_2: %.2f,  Acc_3: %.2f,  Acc_4: %.2f,  Acc_5: %.2f' % (epoch, i, total_losses.avg, accs_1.avg, accs_2.avg, accs_3.avg, accs_4.avg, accs_5.avg)
            print(train_str)
    
    scheduler.step()    # lr scheduler update at the end of each epoch:

    ################################## Validation:
    model.eval()
    requires_grad_(model, False)

    eval_this_epoch = (epoch == 1) or (epoch % 5 == 0) or (epoch>=int(0.75*args.epochs)) # boolean
    if eval_this_epoch:
        val_accs_1, val_accs_2, val_accs_3, val_accs_4, val_accs_5, val_losses_1, val_losses_2, val_losses_3, val_losses_4, val_losses_5, val_total_losses = AverageMeter(), AverageMeter(), AverageMeter(), AverageMeter(), AverageMeter(), AverageMeter(), AverageMeter(), AverageMeter(), AverageMeter(), AverageMeter(), AverageMeter()
            
        for i, (imgs, labels) in enumerate(test_loader_1):
            imgs, labels = imgs.cuda(), labels.cuda()
            logits = model(imgs, 1)    # idx is the num of classes in SOL
            val_loss_1 = loss_fn(logits, labels)
            val_losses_1.append(val_loss_1.item())
            val_accs_1.append((logits.argmax(1) == labels).float().mean().item()*100)

        for i, (imgs, labels) in enumerate(test_loader_2):
            imgs, labels = imgs.cuda(), labels.cuda()
            logits = model(imgs, 2)    # idx is the num of classes in SOL
            val_loss_2 = loss_fn(logits, labels)
            val_losses_2.append(val_loss_2.item())
            val_accs_2.append((logits.argmax(1) == labels).float().mean().item()*100)
        
        for i, (imgs, labels) in enumerate(test_loader_3):
            imgs, labels = imgs.cuda(), labels.cuda()
            logits = model(imgs, 3)    # idx is the num of classes in SOL
            val_loss_3 = loss_fn(logits, labels)
            val_losses_3.append(val_loss_3.item())
            val_accs_3.append((logits.argmax(1) == labels).float().mean().item()*100)
        
        for i, (imgs, labels) in enumerate(test_loader_4):
            imgs, labels = imgs.cuda(), labels.cuda()
            logits = model(imgs, 4)    # idx is the num of classes in SOL
            val_loss_4 = loss_fn(logits, labels)
            val_losses_4.append(val_loss_4.item())
            val_accs_4.append((logits.argmax(1) == labels).float().mean().item()*100)
        
        for i, (imgs, labels) in enumerate(test_loader_5):
            imgs, labels = imgs.cuda(), labels.cuda()
            logits = model(imgs, 5)    # idx is the num of classes in SOL
            val_loss_5 = loss_fn(logits, labels)
            val_losses_5.append(val_loss_5.item())
            val_accs_5.append((logits.argmax(1) == labels).float().mean().item()*100)
              
        ####### Record Data for Curves
        data_plots["train_acc1"].append(accs_1.avg)
        data_plots["train_acc2"].append(accs_2.avg)
        data_plots["train_acc3"].append(accs_3.avg)
        data_plots["train_acc4"].append(accs_4.avg)
        data_plots["train_acc5"].append(accs_5.avg)
        data_plots["train_loss1"].append(losses_1.avg)
        data_plots["train_loss2"].append(losses_2.avg)
        data_plots["train_loss3"].append(losses_3.avg)
        data_plots["train_loss4"].append(losses_4.avg)
        data_plots["train_loss5"].append(losses_5.avg)
        data_plots["train_total_loss"].append(total_losses.avg)
        
        val_total_losses = val_losses_1.avg + val_losses_2.avg + val_losses_3.avg + val_losses_4.avg + val_losses_5.avg
        data_plots["val_acc1"].append(val_accs_1.avg)
        data_plots["val_acc2"].append(val_accs_2.avg)
        data_plots["val_acc3"].append(val_accs_3.avg)
        data_plots["val_acc4"].append(val_accs_4.avg)
        data_plots["val_acc5"].append(val_accs_5.avg)
        data_plots["val_loss1"].append(val_losses_1.avg)
        data_plots["val_loss2"].append(val_losses_2.avg)
        data_plots["val_loss3"].append(val_losses_3.avg)
        data_plots["val_loss4"].append(val_losses_4.avg)
        data_plots["val_loss5"].append(val_losses_5.avg)
        data_plots["val_total_loss"].append(val_total_losses)
        
    if eval_this_epoch:
        val_str = '\n*********** Epoch %d | Test Acc_1: %.2f | Test Acc_2: %.2f | Test Acc_3: %.2f | Test Acc_4: %.2f | Test Acc_5: %.2f\n' % (epoch, val_accs_1.avg, val_accs_2.avg, val_accs_3.avg, val_accs_4.avg, val_accs_5.avg)
        print(val_str)

    # Save MODEL:
    if eval_this_epoch:
        avg_val_TA_list = []
        avg_val_TA_list.append(val_accs_1.avg)
        avg_val_TA_list.append(val_accs_2.avg)
        avg_val_TA_list.append(val_accs_3.avg)
        avg_val_TA_list.append(val_accs_4.avg)
        avg_val_TA_list.append(val_accs_5.avg)
        avg_val_TA = sum(avg_val_TA_list) / len(avg_val_TA_list)    # Averaging performance across all lambdas
        
        if avg_val_TA >= best_TA:
            best_TA = avg_val_TA # update best TA
            torch.save(model.state_dict(), 'UNIFIED_WRN16_8_5_Datasets.pth')

Epoch 1-500 | Loss: 11.3433,  Acc_1: 92.19,  Acc_2: 75.83,  Acc_3: 36.56,  Acc_4: 9.72,  Acc_5: 4.55
Epoch 1-1000 | Loss: 10.0667,  Acc_1: 94.90,  Acc_2: 80.69,  Acc_3: 47.74,  Acc_4: 16.10,  Acc_5: 8.02
Epoch 1-1500 | Loss: 9.2149,  Acc_1: 96.08,  Acc_2: 83.15,  Acc_3: 55.14,  Acc_4: 21.70,  Acc_5: 10.82

*********** Epoch 1 | Test Acc_1: 97.82 | Test Acc_2: 87.96 | Test Acc_3: 69.06 | Test Acc_4: 32.43 | Test Acc_5: 17.54

Epoch 2-500 | Loss: 6.6062,  Acc_1: 98.74,  Acc_2: 89.86,  Acc_3: 76.68,  Acc_4: 40.86,  Acc_5: 21.50
Epoch 2-1000 | Loss: 6.4136,  Acc_1: 98.79,  Acc_2: 89.98,  Acc_3: 77.86,  Acc_4: 42.78,  Acc_5: 22.73
Epoch 2-1500 | Loss: 6.2401,  Acc_1: 98.82,  Acc_2: 90.21,  Acc_3: 78.82,  Acc_4: 44.48,  Acc_5: 24.02
Epoch 3-500 | Loss: 5.5817,  Acc_1: 98.93,  Acc_2: 90.93,  Acc_3: 82.58,  Acc_4: 51.05,  Acc_5: 29.10
Epoch 3-1000 | Loss: 5.5179,  Acc_1: 98.98,  Acc_2: 91.14,  Acc_3: 82.87,  Acc_4: 51.86,  Acc_5: 29.41
Epoch 3-1500 | Loss: 5.4586,  Acc_1: 98.91,  Acc_2: 91.24,

### **Inference**

In [ ]:
checkpoint = torch.load('UNIFIED_WRN16_8_5_Datasets.pth')
model.load_state_dict(checkpoint)

In [16]:
def test_model(model, test_loader, dataset_idx):
    model.eval()
    requires_grad_(model, False)
    test_accs = {}
    test_accs = AverageMeter()

    for i, (imgs, labels) in enumerate(test_loader):
        imgs, labels = imgs.cuda(), labels.cuda()
        logits = model(imgs, dataset_idx)
        test_accs.append((logits.argmax(1) == labels).float().mean().item()*100)
    print(f"Test Accuracy: {test_accs.avg:.2f} %")

In [17]:
test_model(model, test_loader_1, 1)
test_model(model, test_loader_2, 2)
test_model(model, test_loader_3, 3)
test_model(model, test_loader_4, 4)
test_model(model, test_loader_5, 5)

Test Accuracy: 99.66 %
Test Accuracy: 94.44 %
Test Accuracy: 96.00 %
Test Accuracy: 79.41 %
Test Accuracy: 65.19 %
